In [ ]:
#first parse the corpus
# Import the toolkit, giving it a shorter alias 'mptf'
import mptf_parser as mptf

input_folder = r"\MPCD\exports_28-1-2026"
output_folder = r"\MPCD\the syntax project\nounphrase\export_files\conllu_output"

In [2]:
import os
from conllu import parse
from conllu.parser import DEFAULT_FIELD_PARSERS
import mptf_parser as mptf

# ======================================================
# LOAD BOTH CORPORA
# ======================================================

print("\n--- Loading corpora ---")

custom_field_parsers = DEFAULT_FIELD_PARSERS.copy()
custom_field_parsers["id"] = lambda line, i: line[i]
custom_field_parsers["head"] = lambda line, i: line[i]

my_corpus = []
syntactically_annotated_corpus = []

conllu_files = sorted(
    f for f in os.listdir(output_folder)
    if f.lower().endswith(".conllu")
    and "outdated" not in f.lower()
)

for filename in conllu_files:
    file_path = os.path.join(output_folder, filename)

    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = f.read()

    # --- Clean malformed lines ---
    lines = raw_data.splitlines()
    clean_lines = [
        line for line in lines
        if line.startswith("#")
        or line.strip() == ""
        or line.count("\t") == 9
    ]
    clean_data = "\n".join(clean_lines) + "\n"

    # --- Parse sentences ---
    sentences = parse(clean_data, field_parsers=custom_field_parsers)

    for sent in sentences:
        sentence_obj = mptf.Sentence(
            sent,
            source_filename=filename
        )
        sentence_obj.metadata["source_filename"] = filename

        # → always add to the full corpus
        my_corpus.append(sentence_obj)

        # → add to syntactic corpus ONLY if deprel exists
        if any(tok.get("deprel") not in (None, "_") for tok in sent):
            syntactically_annotated_corpus.append(sentence_obj)

# ======================================================
# CONFIRMATION
# ======================================================

print("✔ Corpora loaded successfully.")
print(f"  my_corpus (all texts):")
print(f"    texts:     {len(set(s.metadata['source_filename'] for s in my_corpus))}")
print(f"    sentences: {len(my_corpus)}")

print(f"\n  syntactically_annotated_corpus:")
print(f"    texts:     {len(set(s.metadata['source_filename'] for s in syntactically_annotated_corpus))}")
print(f"    sentences: {len(syntactically_annotated_corpus)}")



--- Loading corpora ---
✔ Corpora loaded successfully.
  my_corpus (all texts):
    texts:     40
    sentences: 31191

  syntactically_annotated_corpus:
    texts:     25
    sentences: 4615


In [ ]:
# ======================================================
# FIND obl:agent TOKENS AND THEIR HEADS
# (Token objects, not dicts)
# ======================================================

from collections import Counter

agent_data = []

for sent in syntactically_annotated_corpus:
    sent_id = sent.sentence_id
    source_file = sent.file_name
    tokens = sent.get_tokens()

    for tok in tokens:
        agent_head_id = None
        is_agent = False

        # --- check basic deprel ---
        if tok.deprel == "obl:agent":
            is_agent = True
            agent_head_id = tok.head

        # --- check enhanced deps (list of tuples) ---
        elif isinstance(tok.deps, list):
            for dep in tok.deps:
                if len(dep) != 2:
                    continue
                rel, head = dep  # tuple order: (relation, head)
                if rel == "obl:agent":
                    is_agent = True
                    agent_head_id = head
                    break

        if not is_agent:
            continue

        # --- agent info ---
        agent_id = tok.id
        agent_lemma = tok.lemma
        agent_form = tok.form

        # --- find head token ---
        head_tok = next((t for t in tokens if t.id == agent_head_id), None)
        head_lemma = head_tok.lemma if head_tok else None
        head_form = head_tok.form if head_tok else None
        head_upos = head_tok.upos if head_tok else None

        # --- find agent-marking adposition ---
        adposition = None
        for child in tokens:
            if child.head == agent_id and child.deprel == "case":
                adposition = child.lemma
                break

        # --- save results ---
        agent_data.append({
            "sentence_id": sent_id,
            "agent_lemma": agent_lemma,
            "agent_form": agent_form,
            "adposition": adposition,
            "head_lemma": head_lemma,
            "head_form": head_form,
            "head_upos": head_upos,
            "source_file": source_file
        })

        # --- debug print ---
        print(f"Found agent: {agent_lemma}, head_id={agent_head_id}, sent_id={sent_id}, adposition={adposition}")

# --- summary ---
print(f"\n✔ Found {len(agent_data)} obl:agent instances.")

# --- distribution of adpositions ---
adpositions = Counter([a["adposition"] for a in agent_data if a["adposition"] is not None])
print("Distribution of agent-marking adpositions:")
for adp, count in adpositions.items():
    print(f"  {adp}: {count}")


Found agent: gayōmart, head_id=13, sent_id=59, adposition=None
Found agent: dūdag, head_id=12, sent_id=192, adposition=az1
Found agent: dūdag, head_id=6, sent_id=197, adposition=az1
Found agent: dūdag, head_id=10, sent_id=206, adposition=az1
Found agent: dūdag, head_id=9, sent_id=213, adposition=az1
Found agent: pusānōš, head_id=15, sent_id=239, adposition=az1

✔ Found 6 obl:agent instances.
Distribution of agent-marking adpositions:
  az1: 5


In [5]:
adp_counter = Counter(entry["adposition"] for entry in agent_data)

print("Distribution of agent-marking adpositions:")
for adp, count in adp_counter.items():
    print(f"  {adp}: {count}")


Distribution of agent-marking adpositions:
  az1: 5


In [6]:
for e in agent_data[:10]:
    print(
        e["sentence_id"],
        "|",
        e["source_file"],
        "|",
        e["adposition"],
        "|",
        e["agent_lemma"],
        "→",
        e["head_lemma"]
    )


192 | MHD-MHDC_mptf.conllu | az1 | dūdag → kirdan
197 | MHD-MHDC_mptf.conllu | az1 | dūdag → kirdan
206 | MHD-MHDC_mptf.conllu | az1 | dūdag → kirdan
213 | MHD-MHDC_mptf.conllu | az1 | dūdag → kirdan
239 | MHD-MHDC_mptf.conllu | az1 | pusānōš → guftan


In [19]:
from collections import Counter

CLAUSAL_HEADS = {"root", "ccomp", "xcomp", "csubj", "advcl", "acl"}
VERB_FORMS = {"Inf", "Vnoun"}  # verbal nouns, infinitives

clause_data = []

for sent in syntactically_annotated_corpus:
    sent_id = sent.sentence_id
    source_file = sent.file_name
    tokens = sent.get_tokens()

    for tok in tokens:
        # --- clause head check ---
        if tok.deprel not in CLAUSAL_HEADS:
            continue

        feats = tok.feats if tok.feats else {}
        verb_form = feats.get("VerbForm")
        subcat = feats.get("Subcat")
        tense = feats.get("Tense")

        # --- Inf/Vnoun or past Part ---
        if verb_form not in VERB_FORMS and not (verb_form == "Part" and tense == "Past"):
            continue

        # --- must be transitive ---
        if subcat != "Tran":
            continue

        # --- collect dependents (numeric heads only) ---
        dependents = [
            t for t in tokens
            if t.head is not None and str(t.head).isdigit() and int(t.head) == int(tok.id)
        ]
        dep_rels = [d.deprel for d in dependents]

        # --- debug: show all dependents ---
        print(f"DEBUG: Sentence {sent_id}, token {tok.id} ({tok.lemma})")
        print(f"  Clause head: {tok.deprel}, VerbForm={verb_form}, Subcat={subcat}, Tense={tense}")
        print(f"  Dependents: {[ (d.lemma,d.deprel) for d in dependents ]}")

        # --- only nsubj/csubj allowed, no obj ---
        if any(r not in ("nsubj", "csubj") for r in dep_rels):
            continue
        if "obj" in dep_rels:
            continue

        # --- collect subjects' lemmata ---
        subject_lemmata = [d.lemma for d in dependents if d.deprel in ("nsubj", "csubj")]

        clause_data.append({
            "sentence_id": sent_id,
            "source_file": source_file,
            "clause_head_lemmata": tok.lemma,
            "subject_lemmata": subject_lemmata,
            "verb_form": verb_form,
            "subcat": subcat,
            "tense": tense
        })

print(f"✔ Found {len(clause_data)} transitive past Inf/Vnoun/Part clause heads with subjects.")

for entry in clause_data[:20]:
    print(f"{entry['sentence_id']} | {entry['source_file']} | {entry['clause_head_lemmata']} → {entry['subject_lemmata']}")


DEBUG: Sentence 6, token 12 (nihuftan)
  Clause head: xcomp, VerbForm=Inf, Subcat=Tran, Tense=None
  Dependents: []
DEBUG: Sentence 17, token 5 (dāštan)
  Clause head: acl, VerbForm=Vnoun, Subcat=Tran, Tense=None
  Dependents: [('grāmīg', 'xcomp')]
DEBUG: Sentence 18, token 7 (handēšīdan)
  Clause head: xcomp, VerbForm=Inf, Subcat=Tran, Tense=None
  Dependents: [('tis1', 'obl'), ('wēš', 'advmod')]
DEBUG: Sentence 19, token 8 (kirdan|šarm kirdan)
  Clause head: xcomp, VerbForm=Inf, Subcat=Tran, Tense=None
  Dependents: [('tis1', 'obl'), ('šarm', 'compound:lvc')]
DEBUG: Sentence 24, token 9 (dāštan)
  Clause head: xcomp, VerbForm=Inf, Subcat=Tran, Tense=None
  Dependents: [('tis1', 'obj'), ('dūr', 'xcomp')]
DEBUG: Sentence 40, token 8 (dāštan)
  Clause head: root, VerbForm=Vnoun, Subcat=Tran, Tense=None
  Dependents: [('tis1', 'obl'), ('mard', 'nsubj:pass'), ('dānāg', 'xcomp')]
DEBUG: Sentence 49, token 10 (dāštan)
  Clause head: root, VerbForm=Inf, Subcat=Tran, Tense=None
  Dependents: 